# MergeSort and Merge (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/MergeSort_java.ipynb)

Chapter 17 — Recursive algorithms. Runnable notebook of `MergeSort` and `Merge`: the algorithms in Java, run on the examples of the chapter, with the «Try it» cell to run them on your own data. The pseudo-code, the examples and the proofs are on the [chapter page](https://fabiofurini.github.io/algorithms/recursion/17-recursive-algorithms/).

The same notebook in [Python](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/MergeSort_python.ipynb) and [C++](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms/MergeSort_cpp.ipynb); all the algorithms of the chapter in the [chapter notebook](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch17_recursive_algorithms_java.ipynb).

**How to run it.** Click the «Open in Colab» badge, then *Runtime → Run all* (Ctrl+F9), or run one cell at a time with Shift+Enter, from top to bottom. The first cell checks that the JDK is there and installs it if it is missing. Each Java program is written to a file by a `%%writefile` cell, then compiled with `javac` and run with `java` in the next cell.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `RecursiveSum` → `recursiveSum` (camelCase).

## Setup

On Colab the JDK is usually already there: the cell below checks it and installs it if it is missing.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Merge Sort and Merge

- **MergeSort**$(a,l,r)$ — *Input*: an array and two indices $l\le r$. *Output*: the sub-array $a[l..r]$ sorted in non-decreasing order. It is called `MergeSort(a, 1, n)` in the pseudo-code, `merge_sort(a, 0, n-1)` in the code.
- **Merge**$(a,l,m,r)$ — *Input*: an array and indices $l\le m<r$ such that $a[l..m]$ and $a[m+1..r]$ are sorted. *Output*: $a[l..r]$ sorted. It copies the two halves into $L$ and $R$ and moves back into $a$, step by step, the smaller of the first two elements not yet copied.

The counter `comparisons` counts the comparisons between elements executed by `Merge`. On the array of the example $a=[9,12,3,16,19,11,12]$ we print the calls of `Merge` in execution order (with the array after each of them, table of the chapter; indices are from 0: adding 1 gives those of the chapter), then the trace of `Merge(a, 0, 1, 3)` on $a=[5,8,2,7]$ (example of the C++ implementation).

In [2]:
%%writefile MergeSort.java
import java.util.*;

public class MergeSort {
    static long comparisons = 0;   // comparisons between elements executed by Merge
    static int msCalls = 0;        // calls of MergeSort

    // merge(a, l, m, r): merges the sorted a[l..m] and a[m+1..r]
    static void merge(int[] a, int l, int m, int r) {
        int n1 = m - l + 1;             // size of the left part
        int n2 = r - m;                 // size of the right part
        int[] L = new int[n1];          // auxiliary arrays
        int[] R = new int[n2];
        for (int i = 0; i < n1; i++) {
            L[i] = a[l + i];
        }
        for (int j = 0; j < n2; j++) {
            R[j] = a[m + 1 + j];
        }
        int i = 0, j = 0, k = l;
        while (i < n1 && j < n2) {      // merge L and R into a[l..r]
            comparisons++;
            if (L[i] <= R[j]) {
                a[k] = L[i];
                i++;
            } else {
                a[k] = R[j];
                j++;
            }
            k++;
        }
        while (i < n1) {                // copy the rest of L
            a[k] = L[i];
            i++;
            k++;
        }
        while (j < n2) {                // copy the rest of R
            a[k] = R[j];
            j++;
            k++;
        }
    }

    // mergeSort(a, l, r): sorts a[l..r]
    static void mergeSort(int[] a, int l, int r) {
        msCalls++;
        if (l < r) {
            int m = l + (r - l) / 2;    // divide
            mergeSort(a, l, m);         // conquer (left half)
            mergeSort(a, m + 1, r);     // conquer (right half)
            merge(a, l, m, r);          // combine
        }
    }

    // ---- variants that print the trace ----
    static void mergeSortLog(int[] a, int l, int r) {
        if (l < r) {
            int m = l + (r - l) / 2;
            mergeSortLog(a, l, m);
            mergeSortLog(a, m + 1, r);
            String left = Arrays.toString(Arrays.copyOfRange(a, l, m + 1));
            String right = Arrays.toString(Arrays.copyOfRange(a, m + 1, r + 1));
            merge(a, l, m, r);
            System.out.println("Merge(a," + l + "," + m + "," + r + ")  " + left + " and " + right + "  ->  " + Arrays.toString(a));
        }
    }

    static void mergeTrace(int[] a, int l, int m, int r) {
        int n1 = m - l + 1, n2 = r - m;
        int[] L = Arrays.copyOfRange(a, l, m + 1), R = Arrays.copyOfRange(a, m + 1, r + 1);
        System.out.println("Merge(a, " + l + ", " + m + ", " + r + ") on a = " + Arrays.toString(a));
        System.out.println("n1=" + n1 + ", n2=" + n2 + ", L=" + Arrays.toString(L) + ", R=" + Arrays.toString(R));
        int i = 0, j = 0, k = l, it = 0;
        while (i < n1 && j < n2) {
            it++;
            String head;
            if (L[i] <= R[j]) {
                head = String.format("%2d. loop 1: L[%d]=%d <= R[%d]=%d? Yes => a[%d] = L[%d] = %d", it, i, L[i], j, R[j], k, i, L[i]);
                a[k] = L[i];
                i++;
            } else {
                head = String.format("%2d. loop 1: L[%d]=%d <= R[%d]=%d? No => a[%d] = R[%d] = %d", it, i, L[i], j, R[j], k, j, R[j]);
                a[k] = R[j];
                j++;
            }
            k++;
            System.out.println(head + ", i=" + i + ", j=" + j + ", k=" + k + "; a=" + Arrays.toString(a));
        }
        while (i < n1) {
            a[k] = L[i];
            System.out.println("    loop 2 (rest of L): a[" + k + "] = L[" + i + "] = " + L[i] + "; a=" + Arrays.toString(a));
            i++;
            k++;
        }
        while (j < n2) {
            a[k] = R[j];
            System.out.println("    loop 3 (rest of R): a[" + k + "] = R[" + j + "] = " + R[j] + "; a=" + Arrays.toString(a));
            j++;
            k++;
        }
        System.out.println("result a = " + Arrays.toString(a) + ", comparisons = " + it);
    }

    public static void main(String[] args) {
        int[] a = {9, 12, 3, 16, 19, 11, 12};
        int n = 7;
        System.out.println("a = " + Arrays.toString(a) + "   n = " + n);
        System.out.println("Calls of Merge in execution order:");
        mergeSortLog(a.clone(), 0, n - 1);
        comparisons = 0;
        msCalls = 0;
        mergeSort(a, 0, n - 1);
        System.out.println("sorted = " + Arrays.toString(a) + "   calls of MergeSort=" + msCalls + " (2n-1=" + (2 * n - 1) + ")   comparisons=" + comparisons);
        System.out.println();
        int[] b = {5, 8, 2, 7};
        mergeTrace(b, 0, 1, 3);
    }
}

Writing MergeSort.java


In [3]:
!javac MergeSort.java && java MergeSort

a = [9, 12, 3, 16, 19, 11, 12]   n = 7
Calls of Merge in execution order:
Merge(a,0,0,1)  [9] and [12]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,2,2,3)  [3] and [16]  ->  [9, 12, 3, 16, 19, 11, 12]
Merge(a,0,1,3)  [9, 12] and [3, 16]  ->  [3, 9, 12, 16, 19, 11, 12]
Merge(a,4,4,5)  [19] and [11]  ->  [3, 9, 12, 16, 11, 19, 12]
Merge(a,4,5,6)  [11, 19] and [12]  ->  [3, 9, 12, 16, 11, 12, 19]
Merge(a,0,3,6)  [3, 9, 12, 16] and [11, 12, 19]  ->  [3, 9, 11, 12, 12, 16, 19]
sorted = [3, 9, 11, 12, 12, 16, 19]   calls of MergeSort=13 (2n-1=13)   comparisons=14

Merge(a, 0, 1, 3) on a = [5, 8, 2, 7]
n1=2, n2=2, L=[5, 8], R=[2, 7]
 1. loop 1: L[0]=5 <= R[0]=2? No => a[0] = R[0] = 2, i=0, j=1, k=1; a=[2, 8, 2, 7]
 2. loop 1: L[0]=5 <= R[1]=7? Yes => a[1] = L[0] = 5, i=1, j=1, k=2; a=[2, 5, 2, 7]
 3. loop 1: L[1]=8 <= R[1]=7? No => a[2] = R[1] = 7, i=1, j=2, k=3; a=[2, 5, 7, 7]
    loop 2 (rest of L): a[3] = L[1] = 8; a=[2, 5, 7, 8]
result a = [2, 5, 7, 8], comparisons = 3


**Try it yourself.** Sort another array: the number of comparisons of `Merge` is at most $n\lceil\log_2 n\rceil$ and the calls of `MergeSort` are always $2n-1$.

In [4]:
%%writefile ProvaMergeSort.java
import java.util.*;

public class ProvaMergeSort {
    public static void main(String[] args) {
        // Sort another array and count the comparisons (at most n*ceil(log2 n)).
        int[] b = {38, 27, 43, 3, 9, 82, 10, 5};
        MergeSort.comparisons = 0;
        MergeSort.msCalls = 0;
        MergeSort.mergeSort(b, 0, b.length - 1);
        System.out.println(Arrays.toString(b) + " " + MergeSort.comparisons + " " + MergeSort.msCalls);
    }
}

Writing ProvaMergeSort.java


In [5]:
!javac ProvaMergeSort.java && java ProvaMergeSort

[3, 5, 9, 10, 27, 38, 43, 82] 17 15


---

Teaching material by [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza University of Rome.
Text, figures and data [CC BY 4.0](https://github.com/fabiofurini/algorithms/blob/main/LICENSE), code [MIT](https://github.com/fabiofurini/algorithms/blob/main/LICENSE-CODE).